# Langchain Agents 

In [16]:
from pydantic import BaseModel
from langchain.agents import create_agent,AgentState
from langgraph.checkpoint.memory import InMemorySaver
from langchain_groq import ChatGroq
from langchain.tools import tool
from dotenv import load_dotenv

load_dotenv()
llm = ChatGroq(
    model="openai/gpt-oss-120b"
)

class CustomAgentState(AgentState):
    user_id: str
    research_topic : str 

@tool
def add(a:float,b: float )-> float:
    """Add two numbers and return the result"""
    try:
        return a+b
    except Exception as error:
        return f"Calculation Failed : {error}"

@tool
def mul(a:float,b: float )-> float:
    """Divide two numbers and return the result"""
    try:
        if b==0: 
            return "Error : Division by zero is not allowed"
        return str(a/b)
    except Exception as error:
        return f"Calculation Failed : {error}"

tools = [add,mul]

agent = create_agent(
    model=llm, 
    tools=tools,
    system_prompt ="You are a helpful math assistant.Use tools for calculations",
    state_schema=CustomAgentState,
    checkpointer=InMemorySaver())

In [ ]:
config = {"configurable":{"thread_id":"user_22"},"recursion_limit":5}

try:
    response = agent.invoke(
        {
        "messages":[
            {"role":"user","content":"Divide 22 by zero"}
        ] ,
        "user_id":"22",
            "research_topic":"Math"
        },
        config=config
    )

    print(response["messages"][-1].content)
except Exception as error: 
    print(f"Agent Execution failed: {error}")

Dividing any non‑zero number by zero is undefined in ordinary arithmetic. There is no finite result for \( \frac{22}{0} \); the expression has no meaning in the real (or complex) number system. If you have a different context in mind (e.g., limits, extended real numbers, or symbolic algebra), let me know and I can discuss that further.


In [3]:
print(response["research_topic"])

Math


In [18]:
for message in response["messages"]:
    print(message)

content='Divide 22 by zero' additional_kwargs={} response_metadata={} id='7edcbcc4-104a-447f-9085-085d5434cd09'
content='Dividing any non‑zero number by zero is undefined in ordinary arithmetic. There is no finite result for \\( \\frac{22}{0} \\); the expression has no meaning in the real (or complex) number system. If you have a different context in mind (e.g., limits, extended real numbers, or symbolic algebra), let me know and I can discuss that further.' additional_kwargs={'reasoning_content': 'The user asks "Divide 22 by zero". That\'s undefined. As a math assistant, we must respond that division by zero is undefined, cannot compute. No tool needed.'} response_metadata={'token_usage': {'completion_tokens': 123, 'prompt_tokens': 169, 'total_tokens': 292, 'completion_time': 0.255260639, 'completion_tokens_details': {'reasoning_tokens': 36}, 'prompt_time': 0.007201074, 'prompt_tokens_details': None, 'queue_time': 0.248087956, 'total_time': 0.262461713}, 'model_name': 'openai/gpt-oss-

In [5]:
for chunk in agent.stream(
    {
       "messages":[
           {"role":"user","content":"Add 150 and 220 ,then multiply the result by 2"}
       ]
    },
    stream_mode=["updates","messages"]
):
    print(chunk)

('messages', (AIMessageChunk(content='', additional_kwargs={}, response_metadata={'model_provider': 'groq'}, id='lc_run--01a210f5-921c-7037-b689-39c0453ded2e', tool_calls=[], invalid_tool_calls=[], tool_call_chunks=[]), {'ls_integration': 'langchain_chat_model', 'langgraph_step': 1, 'langgraph_node': 'model', 'langgraph_triggers': ('branch:to:model',), 'langgraph_path': ('__pregel_pull', 'model'), 'langgraph_checkpoint_ns': 'model:81d3d00b-aa4a-e02a-f160-16e4713659e4', 'checkpoint_ns': 'model:81d3d00b-aa4a-e02a-f160-16e4713659e4', 'ls_provider': 'groq', 'ls_model_name': 'openai/gpt-oss-120b', 'ls_model_type': 'chat', 'ls_temperature': 0.7, 'lc_versions': {'langchain-core': '1.8.4', 'langchain': '1.5.2', 'langchain-groq': '1.2.1'}}))
('messages', (AIMessageChunk(content='', additional_kwargs={'reasoning_content': 'We'}, response_metadata={'model_provider': 'groq'}, id='lc_run--01a210f5-921c-7037-b689-39c0453ded2e', tool_calls=[], invalid_tool_calls=[], tool_call_chunks=[]), {'ls_integra

In [6]:
import asyncio 

async def stream_agent():
    async for event in agent.astream_events(
    {
       "messages":[
           {"role":"user","content":"Add 150 and 220 ,then multiply the result by 2"}
       ]
    },
    version="v2"
    ):
        print("Event:",event["event"],"| name: ",event["name"])
# await stream_agent()